# Connectome-Constrained RNN (Beiran & Litwin-Kumar, 2025)

Teacher–student paradigm of **Beiran & Litwin-Kumar (2025)**, *Prediction of neural activity in connectome-constrained recurrent networks* (Nat. Neurosci.). Teacher and student share the same synaptic weight matrix $J$; the student only trains single-neuron **gains and biases** — implemented with the framework's `gain_rnn` family (`nonlinearity_mode="rate"` + `gain_position="outside"` + softplus, i.e. $r_i = g_i\,\phi(x_i + b_i)$).

- A student trained to match only the teacher's task readout solves the task but does **not** recover single-neuron activity or parameters — the solution space is degenerate.
- Training instead on the recorded activity of $M$ neurons breaks the degeneracy for *unrecorded* neurons once $M$ is large enough.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.stats import linregress

sys.path.append('../src')
from neuralrnn import (AutoConfig, AutoModel, Trainer, TrainingArguments,
                       SupervisedObjective)
from neuralrnn.data.base import BaseDataset


STEPS = dict(teacher_steps=1400, student_steps=7000)
MODEL_DIR = Path(f"./models/16")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE = 'cpu'

# ---- paper hyperparameters (authors' code: fun_lib.py, nn_fig1_Teacher.py) ----
N = 200                    # recurrent units
DTA = 0.1                  # Euler step in units of tau (=1); the paper's Methods say 0.1 but the shipped code uses 0.05
N_TRIALS, BATCH_SIZE = 60, 60
TEACHER_LR, STUDENT_LR = 1e-3, 1e-3
TEACHER_CLIP, STUDENT_CLIP = 5.0, 3.0
NOISE_TEACHER, NOISE_STUDENT = 0.01, 0.002   # injected state noise (std)
SAVE_EVERY = 200           # student checkpoint cadence for error-vs-epoch curves
M_VALUES = [20, 40, 80, 120]

print(f"steps={STEPS}  device={DEVICE}  models -> {MODEL_DIR}")

## 1. The cycling task

Variant of Russo et al. (2018): a transient pulse at $t_I \sim U[2,5]$ in input channel 0 (1) cues an oscillatory readout $\sin(\omega(t-t_I))$ lasting 2 (5) periods, $\omega = 3$. 20% of trials are no-input catch trials. Trial length 20 time units = 400 steps. Ported verbatim from the authors' `create_input` (60 trials, seed 21).

In [ ]:
def create_cycling_input(taus, trials, omega=3.0):
    # Verbatim port of fun_lib.py::create_input (uses the global numpy RNG,
    # seeded by the caller, to stay call-for-call identical to the reference).
    Nt = len(taus)
    input_train = np.zeros((trials, Nt, 2))
    output_train = np.zeros((trials, Nt, 1))
    cond_train = np.zeros((trials))
    To = 2 * np.pi / omega
    for tr in range(trials):
        tI = 2 + 3 * np.random.rand()
        iI = np.argmin(np.abs(taus - tI))
        cond = np.random.randint(2)
        input_train[tr, iI, cond] = 1.
        output_train[tr, iI:, 0] = np.sin(omega * (taus[iI:] - taus[iI]))
        if cond == 0:
            iF = np.argmin(np.abs(taus - tI - To * 2))
            output_train[tr, iF:, 0] = 0.
        else:
            iF = np.argmin(np.abs(taus - tI - To * 5))
            output_train[tr, iF:, 0] = 0.
        if np.random.rand() < 0.2:
            input_train[tr, :, :] = 0.
            output_train[tr, :, 0] = 0
            cond = 3
        cond_train[tr] = cond
    return input_train, output_train, cond_train


class CyclingDataset(BaseDataset):
    kind = "supervised"

    def __init__(self, n_trials=60, dt=0.05, T=12.0, omega=4.0, batch_size=32, seed=21):
        self.taus = np.arange(0, T, dt)
        np.random.seed(seed)
        inputs, targets, conds = create_cycling_input(self.taus, n_trials, omega)
        self.inputs = torch.from_numpy(inputs).float()    # (n_trials, Nt, 2)
        self.targets = torch.from_numpy(targets).float()  # (n_trials, Nt, 1)
        self.conds = conds
        self.mask = torch.ones(n_trials, len(self.taus))
        self.batch_size = batch_size
        self.input_dim, self.output_dim = 2, 1

    def __len__(self):
        return self.inputs.shape[0]

    def __getitem__(self, idx):
        return {"inputs": self.inputs[idx], "targets": self.targets[idx],
                "mask": self.mask[idx]}

    def sample_batch(self):
        idx = torch.randint(0, len(self), (self.batch_size,))
        return {"inputs": self.inputs[idx], "targets": self.targets[idx],
                "mask": self.mask[idx]}

    def first_trial_of(self, cond):
        return int(np.nonzero(self.conds == cond)[0][0])


ds = CyclingDataset(n_trials=N_TRIALS, dt=DTA, batch_size=BATCH_SIZE, seed=21)
taus = ds.taus
print(f"trials={len(ds)}, steps={len(taus)}, "
      f"cond0={int((ds.conds == 0).sum())}, cond1={int((ds.conds == 1).sum())}, "
      f"catch={int((ds.conds == 3).sum())}")

In [ ]:
# Task schematic: one cued trial per condition + one catch trial.
fig, axes = plt.subplots(2, 2, figsize=(10, 4), sharex=True)
for row, (cond, n_per) in enumerate([(0, 2), (1, 5)]):
    tr = ds.first_trial_of(cond)
    axes[row, 0].plot(taus, ds.inputs[tr, :, 0], label="channel 0")
    axes[row, 0].plot(taus, ds.inputs[tr, :, 1], label="channel 1")
    axes[row, 0].set_ylabel("input pulse")
    axes[row, 0].set_title(f"condition {cond}: pulse in channel {cond}")
    axes[row, 1].plot(taus, ds.targets[tr, :, 0], "k")
    axes[row, 1].set_ylabel("target readout")
    axes[row, 1].set_title(f"condition {cond}: sine for {n_per} periods")
axes[0, 0].legend()
for ax in axes[-1]:
    ax.set_xlabel("time (units of tau)")
fig.suptitle("Cycling task")
fig.tight_layout()
plt.show()

## 2. Teacher RNN

Gain-RNN teacher ($N=300$) trained on the cycling task, following `nn_fig1_Teacher.py`:

- **single-neuron params**: $g \sim$ resample-until-positive$(1 + 0.4\mathcal{N})$, $b \sim U(-0.1, 0.1)$;
- **connectivity**: $W = (2.4/\sqrt{N}\,\mathcal{N} + 1/N)$ at 50% density, 70% E / 30% I (I columns ×4), sign-rectified columns; rejection sampling until the Jacobian at the end of a 20-t.u. free run has max Re eig ≤ 1 (the endpoint becomes $h_0$);
- **trained**: $J$, $g$, $b$, $w_{out}$, $h_0$ (Adam lr 1e-3, clip 5, noise 0.01); synapses are re-rectified to their Dale signs after every step via a `post_step_hook` — the framework's built-in `dale` uses $|W|$ semantics, while the reference zeroes sign-violating synapses;
- **input** $w_I \sim \mathcal{N}/\Delta t$ (frozen), so a unit pulse delivers an $\sim\mathcal{N}(0,1)$ kick; all layer biases are zeroed and frozen (the reference model has none).

In [ ]:
def make_teacher_init(N=300, dta=0.05, g0=2.4, sparsity=0.5, percE=0.7,
                      sig_g=0.4, sig_b=0.1, seed=21, verbose=False):
    # Teacher initialization following nn_fig1_Teacher.py (rejection-sampled
    # Dale-compliant sparse connectivity + stability probe for h0).
    np.random.seed(seed)
    nE = int(percE * N)
    softplus = lambda x: np.logaddexp(0.0, x)
    sigmoid = lambda x: 1.0 / (1.0 + np.exp(-x))

    bt = sig_b * (2 * np.random.rand(N) - 1.0)
    gt = sig_g * np.random.randn(N) + 1.0
    while not np.all(gt > 0):
        gt[gt < 0] = sig_g * np.random.randn(int(np.sum(gt < 0))) + 1.0
    wI = np.random.randn(2, N) / dta
    wout = 5.0 * np.random.randn(N, 1) / N

    val, n_try = np.inf, 0
    while val > 1:
        n_try += 1
        Mask_w = (np.random.rand(N, N) > sparsity)
        W = g0 * (1 / np.sqrt(N)) * np.random.randn(N, N) + np.ones((N, N)) / N
        W = W * Mask_w
        W[:, nE:] = 4.0 * W[:, nE:]
        W[:, :nE][W[:, :nE] < 0] = 0.0
        W[:, nE:][W[:, nE:] > 0] = 0.0
        col_sign = np.sign(np.sum(W, axis=0))
        col_sign[col_sign == 0] = 1.0
        refEI = np.tile(col_sign, (N, 1))

        x0 = np.random.randn(N)
        tt = np.arange(0, 20, 0.1)
        xs = np.zeros((len(tt), N))
        xs[0, :] = x0
        for it in range(len(tt) - 1):
            xs[it + 1, :] = xs[it, :] + 0.1 * (
                -xs[it, :] + W.dot(gt * softplus(xs[it, :] + bt)))
        H0 = xs[-1, :]
        Jac = W.dot(np.diag(gt * sigmoid(xs[-1, :] + bt)))
        val = float(np.max(np.real(np.linalg.eigvals(Jac))))
        if verbose:
            print(f"  [teacher init] try {n_try}: max Re eig = {val:.3f}")
    if verbose:
        print(f"  [teacher init] accepted after {n_try} tries")
    return {"J": W.astype(np.float32), "mask": Mask_w.astype(np.float32),
            "refEI": refEI.astype(np.float32), "gt": gt.astype(np.float32),
            "bt": bt.astype(np.float32), "wI": wI.astype(np.float32),
            "wout": wout.astype(np.float32), "H0": H0.astype(np.float32),
            "nE": nE}


def build_gain_rnn(init, *, sigma_rec, trainable_h0, freeze_recurrent,
                   freeze_output, dta=0.05):
    # gain_rnn in the exact [B] configuration: rate mode (state = current x,
    # readout from rates), outside gain r = g*softplus(x+b; beta=1), state
    # noise added post-blend with std sigma_rec (== reference h += sd*xi).
    N = init["J"].shape[0]
    cfg = AutoConfig.for_model(
        "gain_rnn", input_dim=2, latent_dim=N, output_dim=1,
        dt=dta, tau=1.0,
        activation="softplus", activation_params={"beta": 1.0},
        gain_position="outside", nonlinearity_mode="rate",
        sigma_rec=sigma_rec, noise_position="post", noise_alpha_scaling=False,
        rec_mask=init["mask"],
        trainable_h0=trainable_h0, h0_init=init["H0"],
        gain_init=init["gt"], bias_init=init["bt"],
        freeze_input=True, freeze_recurrent=freeze_recurrent,
        freeze_output=freeze_output, freeze_h0=not trainable_h0)
    model = AutoModel.from_config(cfg)
    with torch.no_grad():
        model.input2h.weight.copy_(torch.from_numpy(init["wI"].T.copy()).float())
        model.h2h.weight.copy_(torch.from_numpy(init["J"]).float())
        model.readout_layer.weight.copy_(torch.from_numpy(init["wout"].T.copy()).float())
        model.input2h.bias.zero_()
        model.h2h.bias.zero_()
        model.readout_layer.bias.zero_()
    model.freeze_parameters(patterns=[r"^h2h\.bias$", r"^readout_layer\.bias$"])
    return model


def make_dale_hook(refEI, mask):
    # post_step_hook: w <- relu(w*S)*S * mask (rectify synapses to their Dale
    # signs after each step, as in the paper/reference code).
    S = torch.from_numpy(refEI).float()
    M = torch.from_numpy(mask).float()

    def hook(model):
        with torch.no_grad():
            w = model.h2h.weight
            w.copy_(torch.relu(w * S.to(w.device)) * S.to(w.device) * M.to(w.device))
    return hook


init = make_teacher_init(N=N, dta=DTA, seed=21, verbose=True)
nE = init["nE"]

In [ ]:
# Train the teacher (load-first).
TEACHER_DIR = MODEL_DIR / "teacher"
teacher_history = None
if (TEACHER_DIR / "config.json").exists():
    teacher = AutoModel.from_pretrained(TEACHER_DIR)
    teacher.freeze_parameters(patterns=[r"^h2h\.bias$", r"^readout_layer\.bias$"])
    print(f"Loaded pretrained teacher from {TEACHER_DIR}")
else:
    teacher = build_gain_rnn(init, sigma_rec=NOISE_TEACHER, trainable_h0=True,
                             freeze_recurrent=False, freeze_output=False, dta=DTA)
    args = TrainingArguments(max_steps=STEPS["teacher_steps"],
                             learning_rate=TEACHER_LR,
                             grad_clip_norm=TEACHER_CLIP,
                             log_every=100, seed=0, device=DEVICE)
    trainer = Trainer(teacher, ds, SupervisedObjective(task_type="regression"),
                      args, post_step_hook=make_dale_hook(init["refEI"], init["mask"]))
    teacher_history = trainer.train()
    teacher.save_pretrained(TEACHER_DIR)
teacher.to(DEVICE)

if teacher_history is not None:
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.plot([h["step"] for h in teacher_history],
            [h["loss"] for h in teacher_history], "k")
    ax.set_yscale("log")
    ax.set_xlabel("epoch")
    ax.set_ylabel("readout MSE")
    ax.set_title("Teacher training")
    fig.tight_layout()
    plt.show()

In [ ]:
# Trained teacher: connectivity, single-neuron parameters, task readout.
teacher.eval()
with torch.no_grad():
    out_t_all = teacher(ds.inputs.to(DEVICE))
    readout_t = out_t_all.outputs.cpu()                       # (60, 400, 1)
    rates_t = teacher.get_firing_rates(out_t_all.states).cpu()  # (60, 400, N)

J_t = teacher.h2h.weight.detach().cpu().numpy()
g_t = teacher.gain.detach().cpu().numpy()
b_t = teacher.bias.detach().cpu().numpy()

fig = plt.figure(figsize=(13, 7))
ax0 = fig.add_subplot(2, 3, 1)
im = ax0.imshow(J_t, cmap="bwr", vmin=-0.4, vmax=0.4)
ax0.set_title("teacher connectivity J")
fig.colorbar(im, ax=ax0, fraction=0.046)
ax1 = fig.add_subplot(2, 3, 2)
ax1.plot(np.arange(nE), g_t[:nE], ".", ms=2, c="C0", label="E")
ax1.plot(np.arange(nE, N), g_t[nE:], ".", ms=2, c="C3", label="I")
ax1.axhline(1.0, c="gray", ls="--", lw=0.8)
ax1.set_title("teacher gains")
ax1.legend(markerscale=3)
ax2 = fig.add_subplot(2, 3, 3)
ax2.plot(np.arange(nE), b_t[:nE], ".", ms=2, c="C0")
ax2.plot(np.arange(nE, N), b_t[nE:], ".", ms=2, c="C3")
ax2.axhline(0.0, c="gray", ls="--", lw=0.8)
ax2.set_title("teacher biases")
for j, (cond, n_per) in enumerate([(0, 2), (1, 5)]):
    ax = fig.add_subplot(2, 2, 3 + j)
    tr = ds.first_trial_of(cond)
    ax.plot(taus, ds.targets[tr, :, 0], "k--", lw=1, label="target")
    ax.plot(taus, readout_t[tr, :, 0], c="C0", label="teacher")
    ax.set_title(f"teacher readout — channel {cond} ({n_per} periods)")
    ax.set_xlabel("time (units of tau)")
    ax.legend()
fig.tight_layout()
plt.show()

## 3. student constrained by task output only

The student shares the teacher's $J$, $w_I$, $w_{out}$ and $h_0$ (all frozen) and trains **only gains and biases** to match the teacher's readout (MSE). Initialization: teacher parameters permuted within E/I classes, gains ×0.5 (`nn_fig1_trainStudent.py`). Checkpoints every 200 epochs; readout / activity / gain / bias errors vs the teacher are evaluated per checkpoint (eval mode, no noise). Gray dashed lines: shuffled-identity baselines (100 permutations).

In [ ]:
def permuted_student_gb(gt, bt, nE, g_factor=0.5, seed=0):
    # Student init: teacher params permuted within E/I classes, gains x0.5.
    np.random.seed(seed)
    g, b = gt.copy(), bt.copy()
    e_idx, i_idx = np.arange(nE), np.arange(nE, len(gt))
    g[e_idx] = np.random.permutation(g[e_idx])
    g[i_idx] = np.random.permutation(g[i_idx])
    b[e_idx] = np.random.permutation(b[e_idx])
    b[i_idx] = np.random.permutation(b[i_idx])
    return (g_factor * g).astype(np.float32), b.astype(np.float32)


def build_student(teacher, *, gain_init_s, bias_init_s, sigma_rec=0.002):
    # Connectome-constrained student: shares J / wI / wout / h0 (frozen);
    # only gain and bias are trainable.
    N = teacher.config.latent_dim
    cfg = AutoConfig.for_model(
        "gain_rnn", input_dim=2, latent_dim=N, output_dim=1,
        dt=teacher.config.dt, tau=teacher.config.tau,
        activation="softplus", activation_params={"beta": 1.0},
        gain_position="outside", nonlinearity_mode="rate",
        sigma_rec=sigma_rec, noise_position="post", noise_alpha_scaling=False,
        rec_mask=teacher.rec_mask.detach().cpu().numpy(),
        trainable_h0=False,
        gain_init=gain_init_s, bias_init=bias_init_s,
        freeze_input=True, freeze_recurrent=True,
        freeze_output=True, freeze_h0=True)
    student = AutoModel.from_config(cfg)
    with torch.no_grad():
        student.input2h.weight.copy_(teacher.input2h.weight)
        student.h2h.weight.copy_(teacher.h2h.weight)
        student.readout_layer.weight.copy_(teacher.readout_layer.weight)
        student.h0.copy_(teacher.h0)
        student.input2h.bias.zero_()
        student.h2h.bias.zero_()
        student.readout_layer.bias.zero_()
    return student


@torch.no_grad()
def compute_student_metrics(student, teacher, inputs, teacher_rates,
                            teacher_readout, recorded_idx=None, device="cpu"):
    # Readout/activity/gain/bias errors vs the teacher (no noise); optional
    # recorded/unrecorded split for Fig. 2.
    dev_s = next(student.parameters()).device
    dev_t = next(teacher.parameters()).device
    student.to(device).eval()
    teacher.to(device).eval()
    inputs = inputs.to(device)
    teacher_rates = teacher_rates.to(device)
    teacher_readout = teacher_readout.to(device)
    out = student(inputs)
    rates_s = student.get_firing_rates(out.states)
    met = {"readout_mse": ((out.outputs - teacher_readout) ** 2).mean().item()}
    act_sq = ((rates_s - teacher_rates) ** 2).mean(dim=(0, 1))  # per neuron
    g_sq = (student.gain.detach() - teacher.gain.detach()) ** 2
    b_sq = (student.bias.detach() - teacher.bias.detach()) ** 2
    met["activity_mse"] = act_sq.mean().item()
    met["gain_rmse"] = g_sq.mean().sqrt().item()
    met["bias_rmse"] = b_sq.mean().sqrt().item()
    if recorded_idx is not None:
        rec = np.asarray(recorded_idx)
        unrec = np.setdiff1d(np.arange(len(act_sq)), rec)
        for tag, idx in (("rec", rec), ("unrec", unrec)):
            idx_t = torch.from_numpy(idx)
            met[f"activity_mse_{tag}"] = act_sq[idx_t].mean().item()
            met[f"gain_rmse_{tag}"] = g_sq[idx_t].mean().sqrt().item()
            met[f"bias_rmse_{tag}"] = b_sq[idx_t].mean().sqrt().item()
    student.to(dev_s)
    teacher.to(dev_t)
    return met


def permutation_baselines(teacher_rates, g_t, b_t, n_perm=100, seed=0):
    # Shuffled-identity baselines (Fig. 1d/e gray lines).
    rng = np.random.default_rng(seed)
    r = teacher_rates.detach().cpu().numpy()
    Nn = r.shape[-1]
    act, gain, bias = [], [], []
    for _ in range(n_perm):
        p = rng.permutation(Nn)
        act.append(((r - r[:, :, p]) ** 2).mean())
        gain.append(np.sqrt(((g_t - g_t[p]) ** 2).mean()))
        bias.append(np.sqrt(((b_t - b_t[p]) ** 2).mean()))
    return {"activity_mean": float(np.mean(act)),
            "gain_median": float(np.median(gain)),
            "bias_median": float(np.median(bias))}


def eval_checkpoints(ckpt_dir, teacher, inputs, teacher_rates, teacher_readout,
                     recorded_idx=None):
    ckpt_dir = Path(ckpt_dir)
    steps = sorted(int(d.name.split("-")[1])
                   for d in ckpt_dir.glob("checkpoint-*") if d.is_dir())
    rows = []
    for s in steps:
        model = AutoModel.from_pretrained(ckpt_dir / f"checkpoint-{s}")
        rows.append({"step": s,
                     **compute_student_metrics(model, teacher, inputs,
                                               teacher_rates, teacher_readout,
                                               recorded_idx)})
    return rows


def save_rows(path, rows):
    np.savez(path, **{k: np.array([r[k] for r in rows]) for k in rows[0]})


def load_rows(path):
    z = np.load(path)
    keys = list(z.keys())
    return [{k: float(z[k][i]) for k in keys} for i in range(len(z["step"]))]

In [ ]:
# Fig-1 student: trained on the teacher readout (load-first).
ds_readout = CyclingDataset(n_trials=N_TRIALS, dt=DTA, batch_size=BATCH_SIZE, seed=21)
ds_readout.targets = readout_t.clone()  # targets = noiseless teacher readout

STUDENT0_DIR = MODEL_DIR / "student_M0"
STUDENT0_METRICS = MODEL_DIR / "student_M0_metrics.npz"
if (STUDENT0_DIR / "config.json").exists() and STUDENT0_METRICS.exists():
    student0 = AutoModel.from_pretrained(STUDENT0_DIR)
    rows0 = load_rows(STUDENT0_METRICS)
    print(f"Loaded student_M0 + metrics from {STUDENT0_DIR}")
else:
    g_s, b_s = permuted_student_gb(g_t, b_t, nE, seed=0)
    student0 = build_student(teacher, gain_init_s=g_s, bias_init_s=b_s,
                             sigma_rec=NOISE_STUDENT)
    args = TrainingArguments(max_steps=STEPS["student_steps"],
                             learning_rate=STUDENT_LR,
                             grad_clip_norm=STUDENT_CLIP,
                             log_every=SAVE_EVERY, save_every=SAVE_EVERY,
                             output_dir=str(MODEL_DIR / "ckpt" / "student_M0"),
                             seed=1, device=DEVICE)
    Trainer(student0, ds_readout, SupervisedObjective(task_type="regression"),
            args).train()
    student0.to("cpu")
    student0.save_pretrained(STUDENT0_DIR)
    rows0 = eval_checkpoints(MODEL_DIR / "ckpt" / "student_M0", teacher,
                             ds.inputs, rates_t, readout_t)
    rows0.append({"step": STEPS["student_steps"],
                  **compute_student_metrics(student0, teacher, ds.inputs,
                                            rates_t, readout_t)})
    save_rows(STUDENT0_METRICS, rows0)

baselines = permutation_baselines(rates_t, g_t, b_t, n_perm=100, seed=0)
steps0 = [r["step"] for r in rows0]
print(f"student_M0: final readout_mse={rows0[-1]['readout_mse']:.5f}, "
      f"activity_mse={rows0[-1]['activity_mse']:.4f} "
      f"(shuffled baseline {baselines['activity_mean']:.4f}), "
      f"gain_rmse={rows0[-1]['gain_rmse']:.4f}, bias_rmse={rows0[-1]['bias_rmse']:.4f}")

In [ ]:
# Teacher vs student readout under the two input conditions.
student0.eval()
with torch.no_grad():
    readout_s0 = student0(ds.inputs).outputs.cpu()

fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, cond, n_per in zip(axes, [0, 1], [2, 5]):
    tr = ds.first_trial_of(cond)
    ax.plot(taus, ds.targets[tr, :, 0], "k:", lw=1, label="target")
    ax.plot(taus, readout_t[tr, :, 0], "k", lw=1.2, label="teacher")
    ax.plot(taus, readout_s0[tr, :, 0], c="C1", lw=1, label="student")
    ax.set_title(f"channel {cond} input ({n_per} periods)")
    ax.set_xlabel("time (units of tau)")
axes[0].set_ylabel("readout")
axes[0].legend()
fig.suptitle("Fig. 1 — student reproduces the teacher readout")
fig.tight_layout()
plt.show()

In [ ]:
# Error curves vs training epochs (per-200-epoch checkpoints).
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
axes[0, 0].plot(steps0, [r["readout_mse"] for r in rows0], c="C1")
axes[0, 0].set_yscale("log")
axes[0, 0].set_title("readout error")
axes[0, 1].plot(steps0, [r["activity_mse"] for r in rows0], c="C1")
axes[0, 1].axhline(baselines["activity_mean"], c="gray", ls="--",
                   label="shuffled baseline")
axes[0, 1].set_yscale("log")
axes[0, 1].set_title("activity error (all neurons)")
axes[0, 1].legend()
axes[1, 0].plot(steps0, [r["gain_rmse"] for r in rows0], c="C1")
axes[1, 0].axhline(baselines["gain_median"], c="gray", ls="--",
                   label="shuffled baseline")
axes[1, 0].set_title("gain RMSE")
axes[1, 0].legend()
axes[1, 1].plot(steps0, [r["bias_rmse"] for r in rows0], c="C1")
axes[1, 1].axhline(baselines["bias_median"], c="gray", ls="--",
                   label="shuffled baseline")
axes[1, 1].set_title("bias RMSE")
axes[1, 1].legend()
for ax in axes[-1]:
    ax.set_xlabel("training epochs")
fig.suptitle("Fig. 1 — task readout is learned, single-neuron activity/params are not")
fig.tight_layout()
plt.show()

In [ ]:
# Trained student gain/bias vs teacher: scatter + linear regression.
g_s0 = student0.gain.detach().cpu().numpy()
b_s0 = student0.bias.detach().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, t_par, s_par, name in [(axes[0], g_t, g_s0, "gain"),
                               (axes[1], b_t, b_s0, "bias")]:
    ax.plot(t_par[:nE], s_par[:nE], ".", ms=3, c="C0", label="E")
    ax.plot(t_par[nE:], s_par[nE:], ".", ms=3, c="C3", label="I")
    lr = linregress(t_par, s_par)
    xs = np.linspace(t_par.min(), t_par.max(), 100)
    ax.plot(xs, lr.slope * xs + lr.intercept, "k-", lw=1,
            label=f"slope={lr.slope:.3f}, $R^2$={lr.rvalue ** 2:.3f}")
    lim = [min(t_par.min(), s_par.min()), max(t_par.max(), s_par.max())]
    ax.plot(lim, lim, c="gray", ls="--", lw=0.8)
    ax.set_xlabel(f"teacher {name}")
    ax.set_ylabel(f"student {name}")
    ax.set_title(f"{name}: student vs teacher")
    ax.legend(markerscale=3)
fig.tight_layout()
plt.show()

## 4. students constrained by recorded activity

Students (same shared $J$; only gains/biases trained) are now optimized on the **recorded activity** of $M$ randomly chosen neurons (paper eq. 7: MSE on the recorded rates only — no task term). Recorded sets are the first $M$ entries of one permutation (nested), $M \in \{30, 60, 90, 180\}$. Errors are reported separately for **recorded** and **unrecorded** neurons; the Fig.-1 student serves as the $M=0$ baseline (gray, unrecorded panels only).

In [ ]:
class RecordedActivityObjective:
    # Fig-2 loss: MSE between student and teacher firing rates on the M
    # recorded neurons only (paper eq. 7). No task/readout term.
    def __init__(self, teacher, recorded_idx):
        self.teacher = teacher
        self.recorded_idx = torch.as_tensor(np.asarray(recorded_idx),
                                            dtype=torch.long)

    def compute_loss(self, model, batch):
        out = model(batch["inputs"])
        rates_s = model.get_firing_rates(out.states)
        dev_t = next(self.teacher.parameters()).device
        with torch.no_grad():
            self.teacher.eval()
            out_t = self.teacher(batch["inputs"].to(dev_t))
            rates_t = self.teacher.get_firing_rates(out_t.states)
        idx = self.recorded_idx.to(rates_s.device)
        diff = rates_s[:, :, idx] - rates_t[:, :, idx].to(rates_s.device)
        loss = (diff ** 2).mean()
        return loss, {"loss": loss.item()}


def recorded_sets(M_values, N, seed=0):
    # Nested recorded sets: the first M entries of one permutation.
    perm = np.random.default_rng(seed).permutation(N)
    return {int(M): np.sort(perm[: int(M)]) for M in M_values}


rec_sets = recorded_sets(M_VALUES, N, seed=0)

students, rows_by_M = {}, {}
for k, M in enumerate(M_VALUES):
    sdir = MODEL_DIR / f"student_M{M}"
    mfile = MODEL_DIR / f"student_M{M}_metrics.npz"
    if (sdir / "config.json").exists() and mfile.exists():
        students[M] = AutoModel.from_pretrained(sdir)
        rows_by_M[M] = load_rows(mfile)
        print(f"Loaded student_M{M} + metrics")
        continue
    g_s, b_s = permuted_student_gb(g_t, b_t, nE, seed=10 + k)
    st = build_student(teacher, gain_init_s=g_s, bias_init_s=b_s,
                       sigma_rec=NOISE_STUDENT)
    obj = RecordedActivityObjective(teacher, rec_sets[M])
    args = TrainingArguments(max_steps=STEPS["student_steps"],
                             learning_rate=STUDENT_LR,
                             grad_clip_norm=STUDENT_CLIP,
                             log_every=SAVE_EVERY, save_every=SAVE_EVERY,
                             output_dir=str(MODEL_DIR / "ckpt" / f"student_M{M}"),
                             seed=10 + k, device=DEVICE)
    Trainer(st, ds, obj, args).train()
    st.to("cpu")
    st.save_pretrained(sdir)
    rows = eval_checkpoints(MODEL_DIR / "ckpt" / f"student_M{M}", teacher,
                            ds.inputs, rates_t, readout_t,
                            recorded_idx=rec_sets[M])
    rows.append({"step": STEPS["student_steps"],
                 **compute_student_metrics(st, teacher, ds.inputs, rates_t,
                                           readout_t, recorded_idx=rec_sets[M])})
    save_rows(mfile, rows)
    students[M] = st
    rows_by_M[M] = rows

teacher.to("cpu")
colors = plt.cm.plasma(np.linspace(0, 0.8, len(M_VALUES)))
print("final unrecorded activity MSE:",
      {M: round(rows_by_M[M][-1]["activity_mse_unrec"], 4) for M in M_VALUES})

In [ ]:
# Activity error vs epochs, split recorded / unrecorded.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for M, c in zip(M_VALUES, colors):
    steps = [r["step"] for r in rows_by_M[M]]
    axes[0].plot(steps, [r["activity_mse_rec"] for r in rows_by_M[M]],
                 c=c, label=f"M={M}")
    axes[1].plot(steps, [r["activity_mse_unrec"] for r in rows_by_M[M]],
                 c=c, label=f"M={M}")
axes[1].plot(steps0, [r["activity_mse"] for r in rows0], c="gray", lw=1.2,
             label="M=0 (readout)")
axes[1].axhline(baselines["activity_mean"], c="k", ls=":", lw=1,
                label="shuffled baseline")
axes[0].set_yscale("log")
axes[0].set_title("recorded neurons")
axes[1].set_title("unrecorded neurons")
for ax in axes:
    ax.set_xlabel("training epochs")
    ax.legend()
axes[0].set_ylabel("activity MSE")
fig.suptitle("Fig. 2 — activity error vs epochs")
fig.tight_layout()
plt.show()

In [ ]:
# Gain/bias RMSE vs epochs, split recorded / unrecorded.
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
panels = [("gain_rmse", "gain RMSE", baselines["gain_median"]),
          ("bias_rmse", "bias RMSE", baselines["bias_median"])]
for row, (key, label, base) in enumerate(panels):
    for M, c in zip(M_VALUES, colors):
        steps = [r["step"] for r in rows_by_M[M]]
        axes[row, 0].plot(steps, [r[f"{key}_rec"] for r in rows_by_M[M]],
                          c=c, label=f"M={M}")
        axes[row, 1].plot(steps, [r[f"{key}_unrec"] for r in rows_by_M[M]],
                          c=c, label=f"M={M}")
    axes[row, 1].plot(steps0, [r[key] for r in rows0], c="gray", lw=1.2,
                      label="M=0 (readout)")
    for ax in axes[row]:
        ax.axhline(base, c="k", ls=":", lw=1, label="shuffled baseline")
    axes[row, 0].set_title(f"{label} — recorded")
    axes[row, 1].set_title(f"{label} — unrecorded")
    axes[row, 0].legend()
for ax in axes[-1]:
    ax.set_xlabel("training epochs")
fig.suptitle("Fig. 2 — single-neuron parameter errors are not recovered")
fig.tight_layout()
plt.show()

In [ ]:
# Readout error vs epochs (not split — one scalar readout).
fig, ax = plt.subplots(figsize=(5, 3.5))
for M, c in zip(M_VALUES, colors):
    steps = [r["step"] for r in rows_by_M[M]]
    ax.plot(steps, [r["readout_mse"] for r in rows_by_M[M]], c=c, label=f"M={M}")
ax.set_yscale("log")
ax.set_xlabel("training epochs")
ax.set_ylabel("readout MSE")
ax.set_title("Fig. 2 — readout error vs epochs")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Teacher vs student readout under the two input conditions, per M.
readouts_by_M = {}
for M in M_VALUES:
    students[M].eval()
    with torch.no_grad():
        readouts_by_M[M] = students[M](ds.inputs).outputs.cpu()

fig, axes = plt.subplots(2, len(M_VALUES), figsize=(3.2 * len(M_VALUES), 5),
                         sharey=True)
for j, M in enumerate(M_VALUES):
    for i, (cond, n_per) in enumerate([(0, 2), (1, 5)]):
        ax = axes[i, j]
        tr = ds.first_trial_of(cond)
        ax.plot(taus, readout_t[tr, :, 0], "k", lw=1.1, label="teacher")
        ax.plot(taus, readouts_by_M[M][tr, :, 0], c=colors[j], lw=1,
                label=f"M={M}")
        if i == 0:
            ax.set_title(f"M={M}")
        if j == 0:
            ax.set_ylabel(f"channel {cond} ({n_per} per.)\nreadout")
        if i == 1:
            ax.set_xlabel("time")
axes[0, 0].legend()
fig.suptitle("Fig. 2 — readouts under the two input conditions")
fig.tight_layout()
plt.show()

In [ ]:
# Trained gain/bias vs teacher, per M: recorded neurons colored, unrecorded gray.
fig, axes = plt.subplots(2, len(M_VALUES), figsize=(3.2 * len(M_VALUES), 6.5))
for j, M in enumerate(M_VALUES):
    rec = rec_sets[M]
    unrec = np.setdiff1d(np.arange(N), rec)
    g_sM = students[M].gain.detach().cpu().numpy()
    b_sM = students[M].bias.detach().cpu().numpy()
    for i, (t_par, s_par, name) in enumerate([(g_t, g_sM, "gain"),
                                              (b_t, b_sM, "bias")]):
        ax = axes[i, j]
        ax.plot(t_par[unrec], s_par[unrec], ".", ms=2, c="0.7",
                label="unrecorded")
        ax.plot(t_par[rec], s_par[rec], ".", ms=3, c=colors[j],
                label="recorded")
        # lr = linregress(t_par, s_par)
        lr_unrec = linregress(t_par[unrec], s_par[unrec])
        lr_rec = linregress(t_par[rec], s_par[rec])
        xs = np.linspace(t_par.min(), t_par.max(), 100)
        # ax.plot(xs, lr.slope * xs + lr.intercept, "k-", lw=1, label=f"$R^2$={lr.rvalue ** 2:.2f}")
        ax.plot(xs, lr_unrec.slope * xs + lr_unrec.intercept, c="0.7", lw=1,
                label=f"unrec $R^2$={lr_unrec.rvalue ** 2:.2f}")
        ax.plot(xs, lr_rec.slope * xs + lr_rec.intercept, c=colors[j], lw=1,
                label=f"rec $R^2$={lr_rec.rvalue ** 2:.2f}")
        lim = [min(t_par.min(), s_par.min()), max(t_par.max(), s_par.max())]
        ax.plot(lim, lim, c="gray", ls="--", lw=0.8)
        if i == 0:
            ax.set_title(f"M={M}")
        if j == 0:
            ax.set_ylabel(f"student {name}")
        ax.set_xlabel(f"teacher {name}")
        # if i == 0 and j == 0:
        ax.legend(markerscale=2, fontsize=8, frameon=False)
fig.suptitle("Fig. 2 — single-neuron parameters vs teacher")
fig.tight_layout()
plt.show()

In [ ]:
# Summary: final unrecorded-activity error vs M.
final_unrec = [rows_by_M[M][-1]["activity_mse_unrec"] for M in M_VALUES]
final_rec = [rows_by_M[M][-1]["activity_mse_rec"] for M in M_VALUES]

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(M_VALUES, final_unrec, "o-", c="C0", label="unrecorded (final)")
ax.plot(M_VALUES, final_rec, "s--", c="C2", label="recorded (final)")
ax.axhline(rows0[-1]["activity_mse"], c="gray", ls="--", lw=1.2,
           label="M=0 (readout-trained)")
ax.axhline(baselines["activity_mean"], c="k", ls=":", lw=1,
           label="shuffled baseline")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xticks(M_VALUES)
ax.set_xticklabels([str(M) for M in M_VALUES])
ax.set_xlabel("number of recorded neurons M")
ax.set_ylabel("activity MSE")
ax.set_title("Fig. 2 — recordings break the degeneracy")
ax.legend()
fig.tight_layout()
plt.show()